# Clasificación MNIST con Red Convolucional

MNIST es un ejemplo clásico de reconocimiento de dígitos escritos a mano. Se utiliza la base de datos MNIST, que contiene 60,000 imágenes de entrenamiento y 10,000 imágenes de prueba. Cada imagen es de 28x28 píxeles y cada píxel tiene un valor entre 0 y 255.

Este conjunto de datos marcó un hito en la historia de la IA, con el que [en 1998 el equipo de Yann LeCun utilizó una red neuronal convolucional para lograr una tasa de error del 0.8% en el reconocimiento de dígitos](https://www.youtube.com/watch?v=H0oEr40YhrQ), usando la arquitectura LeNet-5.

Es el mismo ejemplo utilizado para explicar [la teoría de redes neuronales en el video de 3Brown1Blue](https://www.youtube.com/playlist?list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi).

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision import datasets, transforms

## Carga del Dataset

A menudo utilizaremos más de una transformación para preprocesar los datos. Por ejemplo, en el caso de las imágenes, frecuentemente se normalizan y redimensionan. Para hacer esto de manera eficiente, podemos usar la clase `Compose` de `torchvision.transforms`.

In [4]:
# Definir transformaciones para el preprocesamiento de imágenes
# Usamos valores estándar pre-calculados para MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_data = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST('./data', train=False, download=True, transform=transform)

# Crear dataloaders
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

## Definición del Modelo

In [5]:
from torch.nn import functional as F

class CNN(nn.Module): # Definir la red neuronal convolucional
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1) 
    self.pool = nn.MaxPool2d(2, 2)
    self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
    self.fc1 = nn.Linear(7 * 7 * 64, 128)
    self.fc2 = nn.Linear(128, 10)

  def forward(self, x):
    x = self.pool(F.relu(self.conv1(x)))
    x = self.pool(F.relu(self.conv2(x)))
    x = x.view(-1, 7 * 7 * 64) # Aplanar tras las capas convolucionales
    x = F.relu(self.fc1(x))
    x = F.log_softmax(self.fc2(x), dim=1)  # Usar log_softmax para la pérdida de entropía cruzada
    return x
  
model = CNN() # Instanciar la red neuronal

## Entrenamiento del Modelo

### Definición de la Función de Pérdida y el Optimizador

Definimos la función de pérdida y el optimizador. En este caso usaremos el optimizador `optim.Adam`. Adam es una variante del descenso de gradiente estocástico que calcula tasas de aprendizaje individuales para diferentes parámetros.

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

### Entrenamiento

In [7]:
model.train() # Poner el modelo en modo entrenamiento (comportamiento por defecto, pero buena práctica)

for epoch in range(5): # Definir 5 épocas
  
  for i, (images, labels) in enumerate(train_loader):
    # Paso hacia adelante (forward pass)
    outputs = model(images)
    loss = criterion(outputs, labels)

    # Paso hacia atrás y optimización (backward pass)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (i + 1) % 100 == 0:
      print(f'Epoch [{epoch+1}/{10}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}')

Epoch [1/10], Step [100/938], Loss: 0.1112
Epoch [1/10], Step [200/938], Loss: 0.1590
Epoch [1/10], Step [300/938], Loss: 0.0806
Epoch [1/10], Step [400/938], Loss: 0.0185
Epoch [1/10], Step [500/938], Loss: 0.0508
Epoch [1/10], Step [600/938], Loss: 0.1877
Epoch [1/10], Step [700/938], Loss: 0.0895
Epoch [1/10], Step [800/938], Loss: 0.0881
Epoch [1/10], Step [900/938], Loss: 0.0374
Epoch [2/10], Step [100/938], Loss: 0.0664
Epoch [2/10], Step [200/938], Loss: 0.0164
Epoch [2/10], Step [300/938], Loss: 0.0476
Epoch [2/10], Step [400/938], Loss: 0.1172
Epoch [2/10], Step [500/938], Loss: 0.0240
Epoch [2/10], Step [600/938], Loss: 0.0033
Epoch [2/10], Step [700/938], Loss: 0.0036
Epoch [2/10], Step [800/938], Loss: 0.0287
Epoch [2/10], Step [900/938], Loss: 0.1062
Epoch [3/10], Step [100/938], Loss: 0.0528
Epoch [3/10], Step [200/938], Loss: 0.0006
Epoch [3/10], Step [300/938], Loss: 0.0028
Epoch [3/10], Step [400/938], Loss: 0.1009
Epoch [3/10], Step [500/938], Loss: 0.0416
Epoch [3/10

## Evaluación del Modelo

In [8]:
with torch.no_grad():
  correct = 0
  total = 0
  for images, labels in test_loader:
    outputs = model(images)
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()
  print(f'Precisión de la red en las 10000 imágenes de prueba: {100 * correct / total:.2f}%')

Precisión de la red en las 10000 imágenes de prueba: 99.28%


- https://dudeperf3ct.github.io/cnn/mnist/2018/10/17/Force-of-Convolutional-Neural-Networks/